# AI Trading Council 决策实战

本案例展示如何使用多AI模型投票系统进行交易决策。

## 核心功能
- 三模型投票决策（LongCat、讯飞星火、智谱GLM）
- 共识决策生成
- 风险评估
- 决策报告生成

## 1. 环境准备

In [ ]:
import os
import sys
from pathlib import Path

# 添加项目路径
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / 'scripts'))

# 检查环境变量
print("=== AI模型配置检查 ===")
print(f"LongCat API Key: {'已配置' if os.environ.get('LONGCAT_API_KEY') else '未配置'}")
print(f"讯飞星火 API Key: {'已配置' if os.environ.get('SPARK_API_KEY') else '未配置'}")
print(f"智谱GLM API Key: {'已配置' if os.environ.get('GLM_API_KEY') else '未配置'}")

if not os.environ.get('LONGCAT_API_KEY'):
    print("\n⚠️ 警告: LongCat API Key是必需的")
    print("请在.env文件中配置: LONGCAT_API_KEY")

## 2. 初始化AI Council

In [ ]:
from council_engine import CouncilEngine

# 创建Council引擎
engine = CouncilEngine()
print("✅ AI Council初始化成功")
print(f"可用模型: {engine.get_available_models()}")

## 3. 分析单只股票

In [ ]:
# 输入股票代码
stock_code = "600519"  # 贵州茅台
stock_name = "贵州茅台"

print(f"\n=== 分析 {stock_name}({stock_code}) ===")

# 调用AI Council决策
decision = engine.analyze_stock(stock_code, stock_name)

print(f"\n最终决策: {decision.final_decision}")
print(f"置信度: {decision.confidence:.2%}")
print(f"\n决策理由:\n{decision.reasoning}")

## 4. 查看各模型投票详情

In [ ]:
print("\n=== 各模型投票详情 ===")

for vote in decision.votes:
    print(f"\n【{vote.model_name}】 - {vote.role}")
    print(f"  决策: {vote.decision}")
    print(f"  置信度: {vote.confidence:.2%}")
    print(f"  理由: {vote.reasoning[:100]}...")
    print(f"  关键因素: {', '.join(vote.key_factors[:3])}")

## 5. 风险评估

In [ ]:
print("\n=== 风险评估 ===")

if decision.risk_warnings:
    print("⚠️ 风险警告:")
    for warning in decision.risk_warnings:
        print(f"  - {warning}")
else:
    print("✅ 无重大风险")

print(f"\n风险等级: {decision.risk_level}")
print(f"建议仓位: {decision.suggested_position:.1%}")

## 6. 批量分析多只股票

In [ ]:
# 批量分析
stocks = [
    ("600519", "贵州茅台"),
    ("000001", "平安银行"),
    ("300750", "宁德时代"),
]

print("\n=== 批量分析 ===")
results = engine.batch_analyze(stocks)

# 汇总结果
print("\n决策汇总:")
for code, name, decision in results:
    print(f"{name}({code}): {decision.final_decision} (置信度: {decision.confidence:.2%})")

## 7. 生成决策报告

In [ ]:
# 生成报告
report_path = engine.generate_report(decision)

print(f"\n✅ 决策报告已生成: {report_path}")
print("\n报告内容预览:")
print("-" * 50)

# 读取报告
with open(report_path, 'r', encoding='utf-8') as f:
    print(f.read()[:500] + "...")

## 8. 记忆系统（可选）

In [ ]:
# 查看股票历史记忆
memory = engine.get_stock_memory(stock_code)

if memory:
    print(f"\n=== {stock_name} 历史记忆 ===")
    for entry in memory[-5:]:  # 最近5条
        print(f"\n{entry['date']}")
        print(f"  决策: {entry['decision']}")
        print(f"  结果: {entry.get('outcome', '待验证')}")
else:
    print(f"\n{stock_name} 暂无历史记忆")

## 总结

本案例展示了AI Trading Council的完整决策流程：
1. 初始化多模型投票系统
2. 分析单只股票
3. 查看各模型投票详情
4. 风险评估
5. 批量分析
6. 生成决策报告
7. 记忆系统

### 决策等级
- STRONG_BUY: 强烈买入（置信度>0.8）
- BUY: 买入（置信度0.6-0.8）
- HOLD: 持有（置信度0.4-0.6）
- SELL: 卖出（置信度0.2-0.4）
- STRONG_SELL: 强烈卖出（置信度<0.2）

### 模型角色
- LongCat: 量化专家（技术指标、量化因子）
- 讯飞星火: 基本面分析师（财务数据、估值）
- 智谱GLM: 技术分析师（K线形态、趋势）